## 01 — Data Ingestion

> **Automated retrieval of NBA game-level data from the official NBA Stats API.**  
> Traditional and advanced box-score data are fetched season by season, merged, and persisted to disk.  
> Output: per-season CSV files in `data/1_raw/seasons/` and a combined dataset in `data/1_raw/`.

### 1. Setup

In [ ]:
import logging
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT))

import polars as pl
from tqdm.auto import tqdm

from nba_api.stats.endpoints import (
    leaguegamefinder,
    boxscoretraditionalv3,
    boxscoreadvancedv3,
)

# Custom imports
from src.loader import load_config
# Load configuration settings from config.toml
config = load_config()

# Clean logging configuration tailored for Jupyter notebooks and scripts
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
    force=True
)
log = logging.getLogger(__name__)

NOTEBOOK_VERSION = "1.1.0"

log.info(f"Data ingestion notebook v{NOTEBOOK_VERSION} - initialized")

### 2. Helper Functions

In [ ]:
def build_season_list(start_year: int, end_year: int) -> list[str]:
    """Generate a list of NBA season strings in the format 'YYYY-YY' (e.g., '2000-01')."""
    if start_year > end_year:
        raise ValueError(
            f"start_year ({start_year}) must be <= end_year ({end_year})"
            )
    return [
        f"{year}-{str(year + 1)[-2:]}" 
        for year in range(start_year, end_year + 1)
    ]

In [ ]:
def fetch_games_for_season(season: str) -> pl.DataFrame:
    """Fetch NBA games data for a specific season from the NBA API."""
    try:
        response = leaguegamefinder.LeagueGameFinder(
            season_nullable=season,
            season_type_nullable="Regular Season",
        )

        df = (
            pl.from_pandas(response.get_data_frames()[0])
            .with_columns(
                pl.lit(season).alias("SEASON")
            )
        )
        return df
        
    except Exception:
        log.exception(
            f"Error fetching games for season {season}"
        )
        return None

In [ ]:
def extract_unique_game_ids(df_games: pl.DataFrame) -> list[str]:
    """Extract unique game IDs from the games DataFrame."""
    return (
        df_games
        .select("GAME_ID")
        .unique()
        .sort("GAME_ID")
        .to_series()
        .to_list()
    )

In [ ]:
def fetch_boxscore_traditional(game_id: str) -> pl.DataFrame:
    """Fetch NBA boxscore traditional data for a specific game from the NBA API."""
    try:
        response = boxscoretraditionalv3.BoxScoreTraditionalV3(
            game_id=game_id
            )
        return pl.from_pandas(response.get_data_frames()[1])
    
    except Exception:
        log.exception(f"Error fetching boxscore traditional data for game {game_id}")
        return None

In [ ]:
def fetch_boxscore_advanced(game_id: str) -> pl.DataFrame:
    """Fetch NBA boxscore advanced data for a specific game from the NBA API."""
    try:
        response = boxscoreadvancedv3.BoxScoreAdvancedV3(
            game_id=game_id
        )
        return pl.from_pandas(response.get_data_frames()[1])
    
    except Exception:
        log.exception(f"Error fetching boxscore advanced data for game {game_id}")
        return None

In [ ]:
def fetch_single_game(
    game_id: str,
    delay_seconds: float = 1.5
) -> pl.DataFrame:
    """Fetch and process a single game's data from the NBA API."""
    # Fetch traditional boxscore data
    df_traditional = fetch_boxscore_traditional(game_id)
    time.sleep(delay_seconds)

    df_advanced = fetch_boxscore_advanced(game_id)

    if df_traditional is None or df_advanced is None:
        return None
    
    # Merge traditional and advanced data
    df_game = df_traditional.join(
        df_advanced,
        on=["gameId", "teamId"],
        how="inner",
        suffix="_adv",
    )

    return df_game


### 3. Ingestion Pipeline

In [ ]:
# Create directories
RAW_DATA_DIR = Path(config["data"]["raw_data_dir"])
OUTPUT_DIR = ROOT / RAW_DATA_DIR
SEASONS_DIR = OUTPUT_DIR / "seasons"

# Verify directories exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEASONS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def ingest_nba_games(
    seasons: list[str],
    delay_seconds: float = 1.0,
    abort_on_error: bool = True
) -> pl.DataFrame:
    season_frames: list[pl.DataFrame] = []

    for season in seasons:
        season_file = SEASONS_DIR / f"nba_season_{season}.csv"

        if season_file.exists():
            log.info(f"Loading existing season data for {season}")
            season_frames.append(pl.read_csv(season_file, infer_schema_length=0))
            continue

        log.info(f"Ingesting games for season {season}")
        df_games = fetch_games_for_season(season)

        if df_games is None:
            if abort_on_error:
                raise RuntimeError(f"Unable to retrieve game list for season {season}")
            continue

        game_ids = extract_unique_game_ids(df_games)
        log.info(f"Found {len(game_ids)} games for season {season}")
        game_frames: list[pl.DataFrame] = []

        for game_id in tqdm(game_ids, desc=f"Processing {season}"):
            df_game = fetch_single_game(game_id=game_id, delay_seconds=delay_seconds)

            if df_game is None:
                log.warning(f"Skipping game {game_id} (season {season}): no data available")
                continue

            game_frames.append(df_game)

        if not game_frames:
            log.warning(f"No valid games found for season {season}")
            continue

        df_boxscore = pl.concat(game_frames, how="diagonal")
        df_season = df_boxscore.join(
            df_games,
            left_on=["gameId", "teamId"],
            right_on=["GAME_ID", "TEAM_ID"],
            how="left",
        )
        df_season.write_csv(season_file)
        season_frames.append(df_season)
        log.info(f"Ingested {len(df_season)} rows for season {season}")

    if not season_frames:
        raise RuntimeError(f"No valid games found for any season in {seasons}")

    return pl.concat(season_frames, how="diagonal_relaxed")

In [ ]:
def check_missing_games(seasons: list[str]) -> pl.DataFrame:
    """
    Cross-check expected game IDs (from LeagueGameFinder) against
    game IDs actually present in the season CSV files.
    Returns a DataFrame of missing games.
    """
    missing: list[dict] = []

    for season in seasons:
        season_file = SEASONS_DIR / f"nba_season_{season}.csv"

        if not season_file.exists():
            log.warning(f"Season file not found for {season}, skipping check")
            continue

        log.info(f"Checking missing games for season {season}")

        df_expected = fetch_games_for_season(season)
        if df_expected is None:
            log.warning(f"Could not fetch expected game list for season {season}")
            continue

        expected_ids = set(extract_unique_game_ids(df_expected))

        df_saved = pl.read_csv(season_file, infer_schema_length=0)
        actual_ids = set(df_saved["gameId"].unique().to_list())

        missing_ids = expected_ids - actual_ids
        if missing_ids:
            log.warning(f"Season {season}: {len(missing_ids)} missing game(s)")
            for game_id in sorted(missing_ids):
                missing.append({"season": season, "game_id": game_id})
        else:
            log.info(f"Season {season}: all games present")

    return pl.DataFrame(missing, schema={"season": pl.String, "game_id": pl.String})

### 4. Configuration

In [ ]:
# Retrieve settings from config
start_year = config["settings"]["start_season"]
end_year = config["settings"]["end_season"]

delay_seconds = config["settings"]["request_delay"]
abort_on_error = config["settings"]["abort_on_error"]

### 5. Run Ingestion Pipeline

In [ ]:
seasons_to_fetch = build_season_list(
    start_year=start_year,
    end_year=end_year,
)

log.info(f"Fetching data for seasons: {seasons_to_fetch}")

# Run ingestion pipeline
df_games = ingest_nba_games(
    seasons=seasons_to_fetch,
    delay_seconds=delay_seconds,
    abort_on_error=abort_on_error,
)

filename = f"nba_stats_{seasons_to_fetch[0]}_{seasons_to_fetch[-1]}.csv"
output_path = OUTPUT_DIR / filename

df_games.write_csv(output_path)
log.info(
    f"Pipeline executed successfully. {len(df_games)} rows written to: {output_path}"
)

### 6. Missing Games Audit

In [ ]:
# Check for missing games
df_missing = check_missing_games(
    seasons=seasons_to_fetch,
)

missing_filename = f"missing_games_{seasons_to_fetch[0]}_{seasons_to_fetch[-1]}.csv"
missing_path = OUTPUT_DIR / missing_filename

# Save missing games to CSV
df_missing.write_csv(missing_path)
log.info(
    f"Missing games saved to {missing_path}"
)

### 7. Metadata & Lineage Tracking